# 🤖 RAG Chatbot — Google Colab Deployment

**Hướng dẫn:**
1. Vào menu **Runtime → Change runtime type → T4 GPU**
2. Chạy lần lượt từng cell từ trên xuống
3. Upload thư mục project hoặc clone từ GitHub

---

## 1. Kiểm Tra GPU & Môi Trường

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

import psutil
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")

## 2. Upload Project & Cài Dependencies

In [ ]:
# === OPTION A: Upload từ Google Drive ===
# Bước 1: Upload thư mục RAG_chatbot lên Google Drive
# Bước 2: Uncomment và chạy cell bên dưới

from google.colab import drive
drive.mount('/content/drive')

# Đường dẫn tương ứng trên Google Drive
# Thay đổi nếu bạn đặt ở vị trí khác
import shutil, os

DRIVE_PROJECT_PATH = '/content/drive/MyDrive/RAG_chatbot'
LOCAL_PROJECT_PATH = '/content/RAG_chatbot'

if os.path.exists(DRIVE_PROJECT_PATH):
    if os.path.exists(LOCAL_PROJECT_PATH):
        shutil.rmtree(LOCAL_PROJECT_PATH)
    shutil.copytree(DRIVE_PROJECT_PATH, LOCAL_PROJECT_PATH)
    print(f"✅ Copied project to {LOCAL_PROJECT_PATH}")
else:
    print(f"❌ Folder not found: {DRIVE_PROJECT_PATH}")
    print("   Hãy upload thư mục RAG_chatbot lên Google Drive/MyDrive/")

In [ ]:
# === OPTION B: Upload trực tiếp từ máy ===
# Uncomment nếu dùng option B thay vì Google Drive

# from google.colab import files
# uploaded = files.upload()  # Upload file ZIP của project
# !unzip -o RAG_chatbot.zip -d /content/

In [ ]:
# Cài dependencies
%cd /content/RAG_chatbot
!pip install -q -r requirements.txt
print("\n✅ Dependencies installed!")

## 3. Kiểm Tra Config

In [ ]:
%cd /content/RAG_chatbot

import sys
sys.path.insert(0, '/content/RAG_chatbot')

from config import print_config
print_config()

## 4. Crawl Dữ Liệu (nếu chưa có chunks.json)

In [ ]:
import os
if not os.path.exists('/content/RAG_chatbot/data/chunks.json'):
    print("⏳ Crawling data...")
    !python data_crawler.py
else:
    import json
    with open('/content/RAG_chatbot/data/chunks.json', 'r') as f:
        chunks = json.load(f)
    print(f"✅ chunks.json đã tồn tại: {len(chunks)} chunks")

## 5. Build Vector DB (nếu chưa có index)

In [ ]:
import os
if not os.path.exists('/content/RAG_chatbot/indexes/retriever_faiss.index'):
    print("⏳ Building vector database...")
    !python build_vectordb.py --backend faiss
else:
    print("✅ FAISS index đã tồn tại!")

## 6. 🚀 Chạy RAG Pipeline

In [ ]:
import time

# Import pipeline
from rag_pipeline import RAGPipeline

print("⏳ Loading pipeline (lần đầu sẽ tải models từ HuggingFace)...")
start = time.time()

rag = RAGPipeline(
    embedding_key='multilingual-e5-small',
    reranker_key='ms-marco-MiniLM-L6',
    llm_key='qwen2.5-1.5b',
    load_llm=True,
)

print(f"\n✅ Pipeline loaded in {time.time() - start:.1f}s")

In [ ]:
# --- Single Query ---
result = rag.query("What is the transformer architecture and how does self-attention work?", verbose=True)

In [ ]:
# --- Thử thêm câu hỏi ---
questions = [
    "Explain the difference between supervised and unsupervised learning.",
    "How does backpropagation algorithm work in neural networks?",
    "What is retrieval-augmented generation (RAG)?",
]

for q in questions:
    result = rag.query(q, verbose=True)
    print("\n" + "=" * 60)

In [ ]:
# --- Câu hỏi tiếng Việt ---
result_vi = rag.query(
    "Kiến trúc Transformer hoạt động như thế nào?",
    language="vi",
    verbose=True,
)

## 7. 📊 Benchmark

In [ ]:
!python benchmark.py --component embedding

In [ ]:
!python benchmark.py --component retriever

In [ ]:
!python benchmark.py --component reranker

In [ ]:
!python benchmark.py --component llm

## 8. 📈 Evaluation

In [ ]:
# Tạo QA test set (nếu chưa có)
!python evaluate.py --create-dataset

In [ ]:
# Chạy evaluation (10 câu hỏi)
!python evaluate.py --max-questions 10

## 9. 🔬 A/B Testing

In [ ]:
!python ab_testing.py --test embedding --max-questions 5

In [ ]:
!python ab_testing.py --test retrieval --max-questions 5

## 10. 💾 Lưu Kết Quả Về Drive

In [ ]:
# Copy kết quả về Google Drive
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/RAG_chatbot/results'
LOCAL_RESULTS = '/content/RAG_chatbot/results'

if os.path.exists(LOCAL_RESULTS) and os.listdir(LOCAL_RESULTS):
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    for f in os.listdir(LOCAL_RESULTS):
        shutil.copy2(os.path.join(LOCAL_RESULTS, f), DRIVE_RESULTS)
    print(f"✅ Results saved to Google Drive: {DRIVE_RESULTS}")
else:
    print("⚠️ No results to save yet.")